In [1]:
import pandas as pd
import json, xmltodict, time
from pathlib import Path
from itertools import islice

In [2]:
# task specific imports
from Bio import Entrez

In [3]:
# To use the api, NCBI asks we give our emails, it's optional
Entrez.email = "carissa.bleker@nib.si"    

In [4]:
# API limits
MAX_UIDS_PER_LOOKUP = 200  # number of IDs we can put in a single request
ESEARCH_RETMAX = 10000     # number of responses we can request be returned

## File paths

In [5]:
# here we will dump the API lookup results as a JSON
all_sra_details_dump_file = Path("./sra_details_dump.json")

# here we will save the final table
all_sra_details_file = Path("./sra_details.tsv")

## Functions

In [6]:
# https://docs.python.org/3/library/itertools.html#itertools.batched
# If using Python >= 3.12, instead use:
# from itertools import batched

def batched(iterable, n):
    # batched('ABCDEFG', 3) → ABC DEF G
    if n < 1:
        raise ValueError('n must be at least one')
    it = iter(iterable)
    while batch := tuple(islice(it, n)):
        yield batch

## SRA metadata

In [7]:
SRA_STUDY_ACCESSIONS = [
    "ERP015521", 
    "SRP050945", 
    "SRP151817", 
    "DRP005301", 
    "SRP113516", 
    "ERP109042", 
    "SRP279501", 
    "DRP004882", 
    "ERP015703", 
    "SRP315458"   
]

### Attempt to load previously collected data

In [8]:
if all_sra_details_dump_file.exists():
    with open(all_sra_details_dump_file, "r") as f: all_sra_details = json.load(f)
else:
    all_sra_details = {}
print(len(all_sra_details))

0


### Fetch anything missing

In [9]:
failed_sra = []
for study_accession in SRA_STUDY_ACCESSIONS:
    if not (study_accession in all_sra_details):
        
        try:
            study_accession_result = []
            
            
            # Get SRA UIDs from study accession
            handle = Entrez.esearch(db="sra", term=f"{study_accession}[Accession]", field="acc", retmax=ESEARCH_RETMAX)
            study_info = Entrez.read(handle)
            sample_uids = study_info["IdList"]
                   
            # This would probably never happen, but if it does, we need to know, as we'll be missing samples...
            if int(study_info['Count']) > ESEARCH_RETMAX:
                print(f"More than {ESEARCH_RETMAX} samples in {s} study! See https://www.ncbi.nlm.nih.gov/books/NBK25499/#chapter4.retmax on how to deal...")
            
            # Get sample details for each SRA sample
            for _sample_uids in batched(sample_uids, MAX_UIDS_PER_LOOKUP):
                handle = Entrez.esummary(db="sra", id=_sample_uids, retmax=study_info['Count'])
                sample_details = Entrez.read(handle)
                study_accession_result += sample_details

            all_sra_details[study_accession] = study_accession_result
            
            # BioPython handles this, but we can be extra polite API users
            time.sleep(1)

        except Exception as e:
            print(study_accession, e)
            failed_sra.append(study_accession)


In [10]:
# dump what we collected now
with open(all_sra_details_dump_file, "w") as f: json.dump(all_sra_details, f)

In [11]:
# Check what failed
failed_sra

[]

## Prepare metadata table

In [12]:
all_samples = []
for study_accession in all_sra_details:
    for sample_details in all_sra_details[study_accession]:
        
        sample_dict = xmltodict.parse('<xml>'+sample_details['ExpXml']+'</xml>')['xml']
        sample_dict["sample_uid"] = sample_details["Id"]
        sample_dict["study_accession"] = study_accession
        _df = pd.json_normalize(sample_dict, sep='_')
        all_samples.append(_df)

In [13]:
df = pd.concat(all_samples)
print(df.shape)
df.head()

(432, 46)


,Bioproject,Biosample,sample_uid,study_accession,Summary_Title,Summary_Platform_@instrument_model,Summary_Platform_#text,Summary_Statistics_@total_runs,Summary_Statistics_@total_spots,Summary_Statistics_@total_bases,...,Library_descriptor_LIBRARY_LAYOUT_@xmlns:com,Library_descriptor_LIBRARY_LAYOUT_SINGLE,Library_descriptor_LIBRARY_CONSTRUCTION_PROTOCOL_@xmlns:com,Library_descriptor_LIBRARY_CONSTRUCTION_PROTOCOL_#text,Library_descriptor_LIBRARY_NAME,Library_descriptor_LIBRARY_STRATEGY,Library_descriptor_LIBRARY_SOURCE,Library_descriptor_LIBRARY_SELECTION,Library_descriptor_LIBRARY_LAYOUT_PAIRED,Library_descriptor_LIBRARY_CONSTRUCTION_PROTOCOL
0,PRJEB13938,SAMEA3960075,2603987,ERP015521,Illumina HiSeq 2000 sequencing; Leaf transcrip...,Illumina HiSeq 2000,ILLUMINA,1,39805627,4020368327,...,SRA.common,None,SRA.common,Arabidopsis thaliana Col-0 were grown in a gno...,NaN,NaN,NaN,NaN,NaN,NaN
0,PRJEB13938,SAMEA3960074,2603986,ERP015521,Illumina HiSeq 2000 sequencing; Leaf transcrip...,Illumina HiSeq 2000,ILLUMINA,1,41272236,4168495836,...,SRA.common,None,SRA.common,Arabidopsis thaliana Col-0 were grown in a gno...,NaN,NaN,NaN,NaN,NaN,NaN
0,PRJEB13938,SAMEA3960073,2603985,ERP015521,Illumina HiSeq 2000 sequencing; Leaf transcrip...,Illumina HiSeq 2000,ILLUMINA,1,32969766,3329946366,...,SRA.common,None,SRA.common,Arabidopsis thaliana Col-0 were grown in a gno...,NaN,NaN,NaN,NaN,NaN,NaN
0,PRJEB13938,SAMEA3960072,2603984,ERP015521,Illumina HiSeq 2000 sequencing; Leaf transcrip...,Illumina HiSeq 2000,ILLUMINA,1,32685706,3301256306,...,SRA.common,None,SRA.common,Arabidopsis thaliana Col-0 were grown in a gno...,NaN,NaN,NaN,NaN,NaN,NaN
0,PRJEB13938,SAMEA3960070,2603983,ERP015521,Illumina HiSeq 2000 sequencing; Leaf transcrip...,Illumina HiSeq 2000,ILLUMINA,1,49299161,4979215261,...,SRA.common,None,SRA.common,Arabidopsis thaliana Col-0 were grown in a gno...,NaN,NaN,NaN,NaN,NaN,NaN


In [14]:
df.columns

Index(['Bioproject', 'Biosample', 'sample_uid', 'study_accession',
       'Summary_Title', 'Summary_Platform_@instrument_model',
       'Summary_Platform_#text', 'Summary_Statistics_@total_runs',
       'Summary_Statistics_@total_spots', 'Summary_Statistics_@total_bases',
       'Summary_Statistics_@total_size', 'Summary_Statistics_@load_done',
       'Summary_Statistics_@cluster_name', 'Submitter_@acc',
       'Submitter_@center_name', 'Submitter_@contact_name',
       'Submitter_@lab_name', 'Experiment_@acc', 'Experiment_@ver',
       'Experiment_@status', 'Experiment_@name', 'Study_@acc', 'Study_@name',
       'Organism_@taxid', 'Organism_@ScientificName', 'Sample_@acc',
       'Sample_@name', 'Instrument_@ILLUMINA',
       'Library_descriptor_LIBRARY_NAME_@xmlns:com',
       'Library_descriptor_LIBRARY_NAME_#text',
       'Library_descriptor_LIBRARY_STRATEGY_@xmlns:com',
       'Library_descriptor_LIBRARY_STRATEGY_#text',
       'Library_descriptor_LIBRARY_SOURCE_@xmlns:com',
     

In [15]:
## It looks like "Summary_Platform_@instrument_model" is the exact equipment used
df["Summary_Platform_@instrument_model"].value_counts()

Summary_Platform_@instrument_model
Illumina HiSeq 2000             168
Illumina HiSeq 2500             135
HiSeq X Ten                      48
NextSeq 500                      42
Illumina Genome Analyzer IIx     21
Illumina HiSeq 3000              18
Name: count, dtype: int64

In [16]:
## "Summary_Platform_#text" is a more general type of technology, probalby will always be illumina
df["Summary_Platform_#text"].value_counts()

Summary_Platform_#text
ILLUMINA    432
Name: count, dtype: int64

In [17]:
## Probably want to clean the table up here...

# "study_accession" is the identifiers we put in
# "Experiment_@acc" or "Sample_@acc" link to the individual samples, so for example, 
# you may want to do the following before saving the table

# only the columnd we want to keep
df = df[["study_accession", "Experiment_@acc", "Sample_@acc", "Summary_Platform_#text", "Summary_Platform_@instrument_model"]]

# make more useable names
df.columns = ["study_accession", "experiment_accession", "sample_accession", "platform", "platform_instrument_model"]

# take a look
df.head()

,study_accession,experiment_accession,sample_accession,platform,platform_instrument_model
0,ERP015521,ERX1477545,ERS1147209,ILLUMINA,Illumina HiSeq 2000
0,ERP015521,ERX1477544,ERS1147208,ILLUMINA,Illumina HiSeq 2000
0,ERP015521,ERX1477543,ERS1147207,ILLUMINA,Illumina HiSeq 2000
0,ERP015521,ERX1477542,ERS1147206,ILLUMINA,Illumina HiSeq 2000
0,ERP015521,ERX1477541,ERS1147204,ILLUMINA,Illumina HiSeq 2000


In [19]:
# save to table
df.to_csv(all_sra_details_file, sep="\t", index=False)